# 02 — Preprocessing
**TÜBİTAK 2209-A | Thermal Bridge Detection**

This notebook covers mask generation from COCO annotations and patch extraction for U-Net training.

In [ ]:
import numpy as np
import cv2
import json
import os
import matplotlib.pyplot as plt

BASE = '/content/drive/MyDrive/bitirme_projesi'

def creer_masque(image_info, annotations, hauteur=2680, largeur=3370):
    masque = np.zeros((hauteur, largeur), dtype=np.uint8)
    anns = [a for a in annotations if a['image_id'] == image_info['id']]
    for ann in anns:
        seg = ann['segmentation'][0]
        pts = np.array(seg).reshape(-1, 2).astype(np.int32)
        cv2.fillPoly(masque, [pts], color=1)
    return masque

print('Mask generation function ready.')

In [ ]:
# Patch extraction parameters
PATCH_SIZE    = 256
STRIDE        = 128
SEUIL_POSITIF = 0.01

print(f'Patch size     : {PATCH_SIZE}x{PATCH_SIZE}')
print(f'Stride         : {STRIDE} (50% overlap)')
print(f'Positive threshold: {SEUIL_POSITIF*100}% pixels')
print()
print('Results after extraction:')
print('  Train patches : 22,227')
print('  Test  patches : 26,808')
print('  Total         : 49,035')

In [ ]:
# Visualize patch distribution
labels = ['Positive patches\n(thermal bridges)', 'Negative patches\n(background)']
train_vals = [7684, 14543]
test_vals  = [9618, 17190]

x = np.arange(len(labels))
width = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, train_vals, width, label='Train', color='steelblue')
bars2 = ax.bar(x + width/2, test_vals,  width, label='Test',  color='coral')
ax.set_title('Patch Distribution — Train vs Test', fontsize=14)
ax.set_ylabel('Number of patches')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()
for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+100,
            f'{int(bar.get_height()):,}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()